# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Same lane, same mid-panel month (`month = '2026-03'`) as the rest of this week's notebooks.
Before testing any signal, look at the four fields those tests lean on: `total_impressions`,
`avg_position`, `position_volatility`, and `ctr`. The check below is percentile-based rather
than eyeballed off a single histogram, so the heavy tails show up as numbers, not just a shape.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"  # same mid-panel month used across w03/w04 this cycle

raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["ctr"] = (df["total_clicks"] / df["total_impressions"]).round(4)
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["has_clicks"] = (df["total_clicks"] > 0).astype(int)
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)

def tail_summary(series, name):
    s = series.dropna()
    return {
        "field": name, "mean": round(s.mean(), 3), "median": round(s.median(), 3),
        "p90": round(s.quantile(0.90), 3), "p99": round(s.quantile(0.99), 3),
        "max": round(s.max(), 3), "p99_over_median": round(s.quantile(0.99) / max(s.median(), 1e-9), 1),
    }

dist = pd.DataFrame([
    tail_summary(df["total_impressions"], "total_impressions"),
    tail_summary(df["avg_position"], "avg_position"),
    tail_summary(df.loc[df["volatility_is_filled"] == 0, "position_volatility"], "position_volatility"),
    tail_summary(df["ctr"], "ctr"),
])
print("p99_over_median >> 1 flags a heavy right tail — a few pages dominate that field:")
dist

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

p99_over_median >> 1 flags a heavy right tail — a few pages dominate that field:


,field,mean,median,p90,p99,max,p99_over_median
0,total_impressions,1818.437,251.000,4603.000,23735.200,617124.000,94.6
1,avg_position,16.000,8.857,39.321,76.904,262.000,8.7
2,position_volatility,9.413,5.522,23.705,39.755,238.295,7.2
3,ctr,0.004,0.000,0.006,0.053,1.000,52600000.0


## 2. Signal test #1 / #2 / #3 (verdict each)

Three assumptions worth checking before anything gets built on top of them, each scored with a
Spearman correlation (rank-based, so the heavy tails above don't distort it) and mapped to a
verdict:

- **CONFIRMED** — correlation in the expected direction, `|corr| >= 0.15`.
- **OPPOSITE** — correlation the wrong direction, `|corr| >= 0.15`.
- **FALSE** — essentially no relationship, `|corr| < 0.05`.
- **MIXED** — somewhere in between: a real but weak or inconsistent relationship.

1. **Better average position associates with higher CTR.** Sign expected: negative
   (`avg_position` counts up from best, so higher CTR should pair with a *lower* number).
2. **Lower position volatility associates with higher CTR** — the idea that a page holding a
   steady rank earns more trust (and clicks) than one bouncing around. Sign expected: negative.
3. **Pages with zero clicks skew toward worse average positions** than pages with any clicks at
   all. Sign expected: negative (`has_clicks` vs `avg_position`).

In [2]:
def verdict_from_corr(corr, strong=0.15, none=0.05):
    if corr <= -strong:
        return "CONFIRMED"
    if corr >= strong:
        return "OPPOSITE"
    if abs(corr) < none:
        return "FALSE"
    return "MIXED"

def run_signal_test(label, x, y, data):
    d = data.dropna(subset=[x, y])
    corr, pval = spearmanr(d[x], d[y])
    verdict = verdict_from_corr(corr)
    print(f"{label}\n  spearman corr({x}, {y}) = {corr:.3f}, p = {pval:.4g}, n = {len(d):,}\n  verdict: {verdict}\n")
    return corr, pval, verdict

results = {}
results["test1_position_ctr"] = run_signal_test(
    "Test 1 — avg_position vs ctr", "avg_position", "ctr", df)

vol_df = df[df["volatility_is_filled"] == 0]
results["test2_volatility_ctr"] = run_signal_test(
    "Test 2 — position_volatility vs ctr (pages with >1 active day only)",
    "position_volatility", "ctr", vol_df)

results["test3_hasclicks_position"] = run_signal_test(
    "Test 3 — has_clicks vs avg_position", "has_clicks", "avg_position", df)

Test 1 — avg_position vs ctr
  spearman corr(avg_position, ctr) = -0.248, p = 0, n = 151,981
  verdict: CONFIRMED

Test 2 — position_volatility vs ctr (pages with >1 active day only)
  spearman corr(position_volatility, ctr) = -0.370, p = 0, n = 146,356
  verdict: CONFIRMED

Test 3 — has_clicks vs avg_position
  spearman corr(has_clicks, avg_position) = -0.238, p = 0, n = 151,981
  verdict: CONFIRMED



## 3. The flag-linked test

`w04_baseline_score`'s rule discounts priority whenever `HIGH_VOLATILITY` fires — the
assumption being that a page with noisy day-to-day ranking gives a less trustworthy
front-half/back-half trend than a page with a stable rank. This tests that assumption directly:
split content into the same high-volatility (top 10% of `position_volatility`) vs. everyone
else split the rule uses, and compare how spread out `pct_change` is in each group. If the
assumption holds, the high-volatility group's `pct_change` should be noticeably more spread
out — a bigger swing there is more likely noise than a real trend.

In [3]:
def verdict_from_ratio(ratio, strong=1.15):
    if ratio >= strong:
        return "CONFIRMED"
    if ratio <= 1 / strong:
        return "OPPOSITE"
    if abs(ratio - 1) < 0.05:
        return "FALSE"
    return "MIXED"

volatility_p90 = vol_df["position_volatility"].quantile(0.90)
df["high_volatility_flag"] = (
    (df["volatility_is_filled"] == 0) & (df["position_volatility"] >= volatility_p90)
)

high_vol_spread = df.loc[df["high_volatility_flag"], "pct_change"].std()
low_vol_spread = df.loc[~df["high_volatility_flag"], "pct_change"].std()
ratio = high_vol_spread / low_vol_spread
verdict4 = verdict_from_ratio(ratio)

print(f"pct_change std, HIGH_VOLATILITY group:     {high_vol_spread:.3f}")
print(f"pct_change std, everyone else:              {low_vol_spread:.3f}")
print(f"ratio (high-volatility / rest):              {ratio:.2f}")
print(f"verdict: {verdict4}")
results["flag_test_volatility"] = (ratio, None, verdict4)

pct_change std, HIGH_VOLATILITY group:     9.129
pct_change std, everyone else:              20.181
ratio (high-volatility / rest):              0.45
verdict: OPPOSITE


## 4. What this means in practice

Built from the four verdicts above, not asserted ahead of them — read the printed paragraph
after running this in Colab, since the actual verdicts (and therefore the wording) depend on
this month's real numbers.

In [4]:
v1 = results["test1_position_ctr"][2]
v2 = results["test2_volatility_ctr"][2]
v3 = results["test3_hasclicks_position"][2]
v4 = results["flag_test_volatility"][2]

lines = []
lines.append(
    "Position still predicts CTR the way you'd expect" if v1 == "CONFIRMED"
    else "Position alone is a weaker CTR predictor than assumed" if v1 in ("MIXED", "FALSE")
    else "Position and CTR moved the opposite way from what's assumed"
)
lines.append(
    "a steadier rank does track with more clicks" if v2 == "CONFIRMED"
    else "ranking stability isn't a strong stand-in for click performance on its own" if v2 in ("MIXED", "FALSE")
    else "noisier rankings actually paired with more clicks, the opposite of the assumption"
)
lines.append(
    "the HIGH_VOLATILITY discount in the baseline rule is doing what it's meant to" if v4 == "CONFIRMED"
    else "the HIGH_VOLATILITY discount may be flagging pages that aren't actually less predictable" if v4 in ("MIXED", "FALSE")
    else "the HIGH_VOLATILITY discount is backwards for this month's data and is worth revisiting"
)

takeaway = (
    f"For a content team reading the refresh queue: {lines[0]}, {lines[1]}, and {lines[2]}. "
    "Treat every flag on that queue as directional, not proof — this audit only covers one month "
    "and three signals, not the full rule."
)
print(takeaway)

For a content team reading the refresh queue: Position still predicts CTR the way you'd expect, a steadier rank does track with more clicks, and the HIGH_VOLATILITY discount is backwards for this month's data and is worth revisiting. Treat every flag on that queue as directional, not proof — this audit only covers one month and three signals, not the full rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.